# MPStats Pipeline

Рабочий ноутбук для выгрузки, обработки, склейки и классификации MPStats-данных.

Логика шагов вынесена в Python-модули `pipeline/services/`, поэтому здесь остались только параметры и удобные ячейки запуска.


## Как пользоваться

1. Запусти блок **0. Настройки и импорты**.
2. При необходимости открой GUI настроек выгрузки или правил классификации.
3. Запускай нужные шаги ниже как раньше.

Если сырые файлы уже выгружены, чаще всего достаточно запускать шаги 2-6.


## Оглавление

- [0. Настройки и импорты](#step0)
- [1. Выгрузка MPStats](#step1)
- [2. Обогащение файлов](#step2)
- [3. Стандартизация колонок](#step3)
- [4. Парсинг веса и объёма](#step4)
- [5. Склейка файлов](#step5)
- [6. Классификация](#step6)
- [7. SQL-хранилище](#sql)
- [8. Быстрый просмотр результата](#preview)


<a id="step0"></a>
## 0. Настройки и импорты


In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

# Делаем импорт проекта устойчивым, даже если Jupyter открыт не из корня репозитория.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from pipeline.models import PipelinePaths
from pipeline.services.classification_service import classify_file
from pipeline.services.enrich_service import enrich_directory
from pipeline.services.export_service import load_export_settings, run_export
from pipeline.services.merge_service import merge_directory
from pipeline.services.sql_service import (
    export_sql_to_csv,
    import_csv_to_sql,
    import_directory_to_sql,
    sql_load_history,
    sql_query,
    sql_table,
    sql_tables,
)
from pipeline.services.standardize_service import standardize_directory
from pipeline.services.weight_parser_service import parse_weights_directory


In [ ]:
# ====== НАСТРАИВАЕМЫЕ ПАРАМЕТРЫ ======

# Имя текущего отчёта. Попадает в итоговые файлы:
# pipeline/03_<PROJECT_NAME>_merged.csv
# pipeline/03_<PROJECT_NAME>_merged_classified.csv
PROJECT_NAME = "Лимонная кислота 23_04"

# Рабочая папка пайплайна. Обычно менять не нужно.
WORKDIR = PROJECT_ROOT / "pipeline"

# Конфиг шага 1: периоды, cookie, TASKS, фильтры выгрузки.
# Редактируется через GUI ниже или напрямую в JSON.
STEP1_CONFIG_FILE = WORKDIR / "step1_export_config.json"
STEP1_TASKS_ARCHIVE = PROJECT_ROOT / "справочник tasks архив.md"

# Правила классификации.
CLASSIFIER_RULES_FILE = PROJECT_ROOT / "classifiers" / "rules.csv"

# Настройки парсинга веса.
MAX_WEIGHT_KG = 40.0

# Настройки склейки: оставляем только строки min < Продажи, шт < max.
MERGE_MIN_SALES = 0
MERGE_MAX_SALES = 40_000

# Настройки классификации.
CLASSIFIER_WRITE_XLSX = True
CLASSIFIER_FILL_UNCLASSIFIED = {
    "Вид мяса": "Прочие",
    "Подкатегория": "Прочие",
}


# SQL-хранилище. По умолчанию используется локальный DuckDB-файл в корне проекта.
SQL_DB_FILE = PROJECT_ROOT / "mpstats.duckdb"
SQL_TABLE = "mpstats_products"
SQL_IMPORT_MODE = "append"      # append — копить историю, replace — заменить таблицу текущим файлом
SQL_LOAD_NAME = PROJECT_NAME
SQL_EXPORT_QUERY = f"SELECT * FROM {SQL_TABLE}"

# Флаги запуска. Они нужны, чтобы Run All был безопасным.
# Для ручной работы можно запускать ячейки по одной, как раньше.
OPEN_STEP1_GUI = False
RUN_STEP1_EXPORT = False      # осторожно: ходит в MPStats и требует свежий cookie
RUN_STEP2_ENRICH = True
RUN_STEP3_STANDARDIZE = True
RUN_STEP4_PARSE_WEIGHTS = True
RUN_STEP5_MERGE = True
OPEN_CLASSIFIER_GUI = False
RUN_STEP6_CLASSIFY = True
RUN_SQL_IMPORT = False          # осторожно: append повторного файла создаст дубли
RUN_SQL_EXPORT = False

paths = PipelinePaths.create(
    project_root=PROJECT_ROOT,
    workdir=WORKDIR,
    project_name=PROJECT_NAME,
)
paths.ensure_dirs()

DIR_STEP1_RAW = paths.step1_raw_dir
DIR_STEP2_ENRICHED = paths.step2_enriched_dir
DIR_STEP3_STANDARDIZED = paths.step3_standardized_dir
DIR_STEP4_PARSED = paths.step4_parsed_dir
OUT_FILE = paths.merged_csv
CLASSIFIER_OUT_FILE = paths.classified_csv
SQL_IMPORT_FILE = CLASSIFIER_OUT_FILE  # если классификация не нужна, можно поставить OUT_FILE
SQL_EXPORT_FILE = WORKDIR / f"sql_{PROJECT_NAME}_export.csv"

print("[PIPELINE] PROJECT_ROOT          =", PROJECT_ROOT)
print("[PIPELINE] WORKDIR               =", WORKDIR)
print("[PIPELINE] PROJECT_NAME          =", PROJECT_NAME)
print("[PIPELINE] STEP1 raw             =", DIR_STEP1_RAW)
print("[PIPELINE] STEP2 enriched        =", DIR_STEP2_ENRICHED)
print("[PIPELINE] STEP3 standardized    =", DIR_STEP3_STANDARDIZED)
print("[PIPELINE] STEP4 parsed          =", DIR_STEP4_PARSED)
print("[PIPELINE] merged CSV            =", OUT_FILE)
print("[PIPELINE] classified CSV        =", CLASSIFIER_OUT_FILE)


In [ ]:
def show_step_result(step_result):
    """Коротко показывает результат шага и последние детали."""
    print(
        f"[{step_result.name}] ok={step_result.ok} "
        f"skipped={step_result.skipped} errors={step_result.errors} "
        f"rows={step_result.rows} output={step_result.output}"
    )
    if step_result.details:
        display(pd.DataFrame(step_result.details).tail(20))
    return step_result


<a id="step1"></a>
## 1. Выгрузка MPStats


Периоды, cookie, список задач и `filterModel` редактируются в `pipeline/step1_export_config.json` через GUI. В ноутбуке оставлены только путь к конфигу и кнопки запуска.


In [ ]:
def launch_step1_gui() -> None:
    cmd = [
        sys.executable,
        "-m",
        "pipeline.step1_gui",
        "--config",
        str(STEP1_CONFIG_FILE),
        "--archive",
        str(STEP1_TASKS_ARCHIVE),
    ]
    subprocess.Popen(cmd, cwd=str(PROJECT_ROOT))
    print("[STEP 1] GUI launched:", " ".join(cmd))

print("[STEP 1] CONFIG_FILE   =", STEP1_CONFIG_FILE)
print("[STEP 1] TASKS_ARCHIVE =", STEP1_TASKS_ARCHIVE)
print("[STEP 1] Для открытия GUI выполни: launch_step1_gui()")


In [ ]:
# Открыть GUI настроек выгрузки.
# Поставь OPEN_STEP1_GUI = True в настройках или вызови launch_step1_gui() вручную.
if OPEN_STEP1_GUI:
    launch_step1_gui()
else:
    print("[STEP 1] GUI не открыт. Для открытия: OPEN_STEP1_GUI=True или launch_step1_gui()")


In [ ]:
# Запустить выгрузку MPStats по текущему step1 config.
# Важно: в STEP1_CONFIG_FILE должен быть актуальный cookie.
if RUN_STEP1_EXPORT:
    export_settings = load_export_settings(STEP1_CONFIG_FILE, default_save_dir=DIR_STEP1_RAW)
    step1_result = run_export(export_settings, log_dir=paths.logs_dir)
    show_step_result(step1_result)
else:
    print("[STEP 1] выгрузка выключена. Для запуска поставь RUN_STEP1_EXPORT=True.")


<a id="step2"></a>
## 2. Обогащение файлов


Добавляет в сырые CSV дату, маркетплейс и категорию по имени файла.


In [ ]:
if RUN_STEP2_ENRICH:
    step2_result = enrich_directory(DIR_STEP1_RAW, DIR_STEP2_ENRICHED)
    show_step_result(step2_result)
else:
    print("[STEP 2] выключен флагом RUN_STEP2_ENRICH=False")


<a id="step3"></a>
## 3. Стандартизация колонок


Приводит Ozon/WB/Яндекс.Маркет к единому канону колонок.


In [ ]:
if RUN_STEP3_STANDARDIZE:
    step3_result = standardize_directory(DIR_STEP2_ENRICHED, DIR_STEP3_STANDARDIZED)
    show_step_result(step3_result)
else:
    print("[STEP 3] выключен флагом RUN_STEP3_STANDARDIZE=False")


<a id="step4"></a>
## 4. Парсинг веса и объёма


Парсит вес из `Название`, считает объём, год и цену за кг. Главный параметр здесь — `MAX_WEIGHT_KG` в блоке настроек.


In [ ]:
if RUN_STEP4_PARSE_WEIGHTS:
    step4_result = parse_weights_directory(
        DIR_STEP3_STANDARDIZED,
        DIR_STEP4_PARSED,
        max_weight_kg=MAX_WEIGHT_KG,
    )
    show_step_result(step4_result)
else:
    print("[STEP 4] выключен флагом RUN_STEP4_PARSE_WEIGHTS=False")


<a id="step5"></a>
## 5. Склейка файлов


Склеивает CSV после шага 4, фильтрует продажи и пишет итоговый merged CSV.


In [ ]:
if RUN_STEP5_MERGE:
    result, step5_result = merge_directory(
        DIR_STEP4_PARSED,
        OUT_FILE,
        min_sales=MERGE_MIN_SALES,
        max_sales=MERGE_MAX_SALES,
    )
    show_step_result(step5_result)
    print("[STEP 5] result shape:", result.shape)
else:
    print("[STEP 5] выключен флагом RUN_STEP5_MERGE=False")


<a id="step6"></a>
## 6. Классификация


Применяет правила из `CLASSIFIER_RULES_FILE`, сохраняет CSV и опционально XLSX.


In [ ]:
def launch_classifier_gui() -> None:
    cmd = [sys.executable, "-m", "classifiers.gui"]
    subprocess.Popen(cmd, cwd=str(PROJECT_ROOT))
    print("[STEP 6] GUI launched:", " ".join(cmd))

print("[STEP 6] RULES_FILE =", CLASSIFIER_RULES_FILE)
print("[STEP 6] OUT_FILE   =", CLASSIFIER_OUT_FILE)
print("[STEP 6] Для открытия GUI правил выполни: launch_classifier_gui()")


In [ ]:
# Открыть GUI правил классификации.
# Поставь OPEN_CLASSIFIER_GUI = True в настройках или вызови launch_classifier_gui() вручную.
if OPEN_CLASSIFIER_GUI:
    launch_classifier_gui()
else:
    print("[STEP 6] GUI не открыт. Для открытия: OPEN_CLASSIFIER_GUI=True или launch_classifier_gui()")


In [ ]:
if RUN_STEP6_CLASSIFY:
    result, cls_report, step6_result = classify_file(
        OUT_FILE,
        CLASSIFIER_OUT_FILE,
        rules_path=CLASSIFIER_RULES_FILE,
        write_xlsx=CLASSIFIER_WRITE_XLSX,
        fill_unclassified=CLASSIFIER_FILL_UNCLASSIFIED,
    )
    show_step_result(step6_result)
    if len(cls_report) > 0:
        display(
            cls_report.sort_values(["applied_rows", "candidate_rows"], ascending=False)
            .reset_index(drop=True)
            .head(20)
        )
    print("[STEP 6] result shape:", result.shape)
else:
    print("[STEP 6] выключен флагом RUN_STEP6_CLASSIFY=False")


<a id="sql"></a>
## 7. SQL-хранилище


Здесь можно загрузить итоговый CSV в локальную DuckDB-базу, посмотреть таблицы/историю загрузок и выгрузить SQL-запрос обратно в CSV.

Обычный сценарий: после шага 6 поставь `RUN_SQL_IMPORT = True` и запусти импорт. Для повторной загрузки того же файла лучше использовать `SQL_IMPORT_MODE = "replace"`, иначе `append` добавит строки ещё раз.


In [ ]:
print("[SQL] DB_FILE     =", SQL_DB_FILE)
print("[SQL] TABLE       =", SQL_TABLE)
print("[SQL] IMPORT_FILE =", SQL_IMPORT_FILE)
print("[SQL] EXPORT_FILE =", SQL_EXPORT_FILE)
print("[SQL] IMPORT_MODE =", SQL_IMPORT_MODE)


In [ ]:
if RUN_SQL_IMPORT:
    source_file = SQL_IMPORT_FILE if Path(SQL_IMPORT_FILE).exists() else OUT_FILE
    sql_import_result = import_csv_to_sql(
        source_file,
        db_path=SQL_DB_FILE,
        table_name=SQL_TABLE,
        mode=SQL_IMPORT_MODE,
        load_name=SQL_LOAD_NAME,
        project_name=PROJECT_NAME,
    )
    show_step_result(sql_import_result)
else:
    print("[SQL] импорт выключен. Для загрузки в БД поставь RUN_SQL_IMPORT=True.")


In [ ]:
# Таблицы и история загрузок в базе.
display(sql_tables(SQL_DB_FILE))
display(sql_load_history(SQL_DB_FILE).head(20))


In [ ]:
# Быстрый просмотр данных из SQL.
# Можно менять запрос под нужную выборку.
try:
    sql_preview = sql_table(SQL_DB_FILE, SQL_TABLE, limit=30)
    display(sql_preview)
except Exception as exc:
    print(f"[SQL] пока не удалось прочитать таблицу {SQL_TABLE!r}: {exc}")


In [ ]:
if RUN_SQL_EXPORT:
    sql_export_result = export_sql_to_csv(
        db_path=SQL_DB_FILE,
        output_file=SQL_EXPORT_FILE,
        query=SQL_EXPORT_QUERY,
    )
    show_step_result(sql_export_result)
else:
    print("[SQL] экспорт выключен. Для выгрузки CSV из SQL поставь RUN_SQL_EXPORT=True.")


<a id="preview"></a>
## 8. Быстрый просмотр результата


In [ ]:
# Можно запускать после шага 5 или 6.
if "result" not in globals():
    preview_file = CLASSIFIER_OUT_FILE if CLASSIFIER_OUT_FILE.exists() else OUT_FILE
    result = pd.read_csv(preview_file, sep=";", encoding="utf-8-sig", low_memory=False)
    print("[PREVIEW] loaded:", preview_file)

preview_columns = [
    "Дата", "Маркетплейс", "Категория", "SKU", "Бренд", "Название",
    "Продажи, шт", "Продавец", "Средняя цена, руб", "Выручка, руб",
    "Вес, кг", "Объем, т", "Год", "Цена за кг",
]
existing_columns = [column for column in preview_columns if column in result.columns]
preview = result[existing_columns].copy() if existing_columns else result.copy()
if "Продажи, шт" in preview.columns:
    preview = preview.sort_values(by="Продажи, шт", ascending=False)

display(preview.head(30))
print("[PREVIEW] rows=", len(result), "cols=", len(result.columns))
